# 08 - Modelado Baseline

## Objetivo

Este notebook implementa la primera etapa de modelado del proyecto.

El objetivo es evaluar si las features construidas en el Notebook 06 contienen información suficiente para superar modelos de referencia simples.

Para ello se comparan tres enfoques:

- Baseline de media histórica.
- Baseline naive basado en el retorno reciente de BTC.
- Modelo lineal base.

El propósito no es generar señales de trading ni recomendaciones de inversión, sino medir de forma reproducible si existe capacidad predictiva inicial sobre el retorno diario futuro de BTC.

## Alcance

Este notebook pertenece a la Fase 1 del pipeline.

Utiliza exclusivamente el dataset generado en el Notebook 06, compuesto por features de Fase 1 y el target `target_btc_return_t1`.

La expansión de features, los modelos avanzados y el análisis final de hipótesis se desarrollan posteriormente en el Notebook 09.

# 1. Librerías y configuración

Se importan las librerías necesarias para cargar el dataset, realizar el split temporal, entrenar el modelo lineal y calcular métricas de evaluación.

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 2. Carga del dataset de modelado

Se carga el dataset generado en el Notebook 06, compuesto por las features de Fase 1 y el target del modelo.

Este dataset será utilizado para construir los baselines y entrenar el modelo lineal base.

In [2]:
PROJECT_PATH = r"C:\DS2_BTC_DXY_ORO_VIX"

dataset_path = os.path.join(
    PROJECT_PATH,
    "data",
    "processed",
    "dataset_modeling.csv"
)

In [3]:
modeling_df = pd.read_csv(
    dataset_path,
    index_col=0,
    parse_dates=True
)

print("=== DATASET DE MODELADO CARGADO ===")
print(f"Shape: {modeling_df.shape}")
print(f"Rango temporal: {modeling_df.index.min().date()} → {modeling_df.index.max().date()}")

=== DATASET DE MODELADO CARGADO ===
Shape: (744, 8)
Rango temporal: 2023-05-17 → 2026-05-05


# 3. Validación inicial del dataset

Se valida la estructura general del dataset de modelado antes de definir las variables explicativas y el target.

Esta revisión permite confirmar que el archivo fue cargado correctamente, que conserva el orden temporal y que no contiene valores faltantes.

In [4]:
print("=== VALIDACIÓN INICIAL DEL DATASET ===")
print()

print(f"Shape: {modeling_df.shape}")
print(f"Rango temporal: {modeling_df.index.min().date()} → {modeling_df.index.max().date()}")
print(f"Orden cronológico correcto: {modeling_df.index.is_monotonic_increasing}")
print(f"Duplicados temporales: {modeling_df.index.duplicated().sum()}")

print()
print("NaN por columna:")
print(modeling_df.isna().sum())

=== VALIDACIÓN INICIAL DEL DATASET ===

Shape: (744, 8)
Rango temporal: 2023-05-17 → 2026-05-05
Orden cronológico correcto: True
Duplicados temporales: 0

NaN por columna:
target_btc_return_t1      0
btc_return_lag1           0
btc_return_lag2           0
btc_volatility_7d_lag1    0
btc_ma_7d_lag1            0
dxy_return_lag1           0
gold_return_lag1          0
vix_close_lag1            0
dtype: int64


## Conclusiones

El dataset de modelado fue cargado correctamente y presenta una estructura consistente con las decisiones de diseño definidas en el Notebook 06.

Las observaciones mantienen el orden cronológico, no se detectaron registros duplicados y ninguna de las variables contiene valores faltantes.

Por lo tanto, el dataset se considera apto para avanzar hacia la definición de las variables explicativas y el target del modelo.

# 4. Definición de variables explicativas y target

Se separan las variables explicativas (features) y la variable objetivo (target).

Las features corresponden exclusivamente a las variables construidas durante la Fase 1 del proyecto y respetan la regla temporal definida en el pipeline:

Features ≤ t-1

Target = t+1

Esta separación garantiza la ausencia de leakage temporal durante el entrenamiento y la evaluación de los modelos.

In [5]:
X = modeling_df.drop(columns=["target_btc_return_t1"])

y = modeling_df["target_btc_return_t1"]

In [6]:
print("=== FEATURES ===")
print(X.columns)

print()

print("=== TARGET ===")
print(y.name)

print()

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")

=== FEATURES ===
Index(['btc_return_lag1', 'btc_return_lag2', 'btc_volatility_7d_lag1',
       'btc_ma_7d_lag1', 'dxy_return_lag1', 'gold_return_lag1',
       'vix_close_lag1'],
      dtype='str')

=== TARGET ===
target_btc_return_t1

Shape X: (744, 7)
Shape y: (744,)


# 5. División temporal Train-Test

Se divide el dataset en conjuntos de entrenamiento y prueba respetando el orden cronológico de las observaciones.

A diferencia de otros problemas de Machine Learning, las series temporales requieren preservar la secuencia de los datos, ya que mezclar observaciones del futuro con observaciones del pasado introduciría leakage temporal.

Se adopta una división aproximada del 70% para entrenamiento y 30% para prueba.

In [7]:
split_index = int(len(X) * 0.7)

print(f"Punto de corte: {split_index}")

Punto de corte: 520


In [8]:
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [9]:
print("=== TRAIN ===")
print(X_train.shape)
print(y_train.shape)

print()

print("=== TEST ===")
print(X_test.shape)
print(y_test.shape)

=== TRAIN ===
(520, 7)
(520,)

=== TEST ===
(224, 7)
(224,)


## Conclusiones

La separación entre entrenamiento y prueba se realizó respetando el orden temporal de las observaciones.

De este modo, los modelos serán entrenados exclusivamente con información del pasado y evaluados sobre datos posteriores, reproduciendo de manera más realista las condiciones bajo las cuales se utilizarían en un escenario de predicción.

# 6. Baseline de media histórica

Se construye un primer modelo de referencia basado en la media histórica de los retornos observados durante el conjunto de entrenamiento.

Este baseline representa una estrategia extremadamente simple: asumir que el retorno futuro será igual al retorno promedio observado en el pasado.

Su función no es generar predicciones precisas, sino proporcionar un punto de comparación mínimo para evaluar si los modelos posteriores son capaces de extraer información adicional de las variables explicativas.

In [10]:
historical_mean = y_train.mean()

print("Media histórica:")
print(historical_mean)

Media histórica:
0.002709240478158523


In [11]:
baseline_mean_pred = np.full(
    shape=len(y_test),
    fill_value=historical_mean
)

In [12]:
print(baseline_mean_pred[:10])

[0.00270924 0.00270924 0.00270924 0.00270924 0.00270924 0.00270924
 0.00270924 0.00270924 0.00270924 0.00270924]


## 6.1. Evaluación del baseline

Se evalúa el desempeño del baseline de media histórica sobre el conjunto de prueba.

Para ello se utilizan tres métricas de regresión:

- Error absoluto medio (MAE).
- Raíz del error cuadrático medio (RMSE).
- Coeficiente de determinación (R²).

Estas métricas servirán como referencia para comparar posteriormente el baseline naive y el modelo lineal.

In [13]:
mae_mean = mean_absolute_error(
    y_test,
    baseline_mean_pred
)

In [14]:
rmse_mean = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_mean_pred
    )
)

In [15]:
r2_mean = r2_score(
    y_test,
    baseline_mean_pred
)

In [16]:
print("=== BASELINE MEDIA HISTÓRICA ===")
print()

print(f"MAE: {mae_mean:.6f}")
print(f"RMSE: {rmse_mean:.6f}")
print(f"R²: {r2_mean:.6f}")

=== BASELINE MEDIA HISTÓRICA ===

MAE: 0.018624
RMSE: 0.025780
R²: -0.016449


## Interpretación preliminar

El baseline de media histórica presenta errores del orden de algunos puntos porcentuales y un coeficiente de determinación prácticamente nulo.

Este comportamiento es consistente con la naturaleza extremadamente simple del modelo, cuya única estrategia consiste en asumir que el retorno futuro será igual al retorno promedio observado en el conjunto de entrenamiento.

Las métricas obtenidas constituyen una referencia mínima que permitirá evaluar si modelos más elaborados son capaces de extraer información adicional de las variables explicativas.

# 7. Baseline Naive

Se implementa un segundo modelo de referencia basado en la persistencia de los retornos.

Este enfoque asume que el mejor estimador para el retorno futuro es simplemente el retorno observado en la jornada inmediatamente anterior.

A diferencia del baseline de media histórica, este modelo incorpora información reciente del mercado, aunque sin aprender relaciones entre variables.

Su desempeño permitirá establecer una referencia más exigente para evaluar posteriormente el modelo lineal.

In [17]:
baseline_naive_pred = X_test["btc_return_lag1"]

In [18]:
print(baseline_naive_pred.head())

Date
2025-06-13   -0.025372
2025-06-16    0.001529
2025-06-17    0.011793
2025-06-18   -0.020559
2025-06-20    0.002698
Name: btc_return_lag1, dtype: float64


## 7.1. Evaluación del baseline naive

Se evalúa el desempeño del modelo de persistencia utilizando las mismas métricas aplicadas anteriormente.

Esto permitirá comparar directamente ambos modelos de referencia y determinar si la información más reciente del mercado aporta ventajas respecto de asumir un retorno promedio constante.

In [19]:
mae_naive = mean_absolute_error(
    y_test,
    baseline_naive_pred
)

rmse_naive = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_naive_pred
    )
)

r2_naive = r2_score(
    y_test,
    baseline_naive_pred
)

In [20]:
print("=== BASELINE NAIVE ===")
print()

print(f"MAE: {mae_naive:.6f}")
print(f"RMSE: {rmse_naive:.6f}")
print(f"R²: {r2_naive:.6f}")

=== BASELINE NAIVE ===

MAE: 0.028166
RMSE: 0.037965
R²: -1.204368


## Interpretación

El baseline naive presentó un desempeño considerablemente inferior al baseline de media histórica en las tres métricas consideradas.

Estos resultados sugieren que los retornos diarios de BTC no exhiben una persistencia simple entre observaciones consecutivas. En particular, asumir que el retorno futuro repetirá el comportamiento reciente produjo errores mayores que utilizar un valor promedio constante.

Por lo tanto, la incorporación de memoria de muy corto plazo, considerada de manera aislada, no parece aportar capacidad predictiva sobre el retorno diario futuro de BTC.

# 8. Modelo de regresión lineal

Se entrena un modelo de regresión lineal utilizando las siete variables explicativas construidas durante la Fase 1.

A diferencia de los modelos baseline, la regresión lineal intenta aprender relaciones entre las variables disponibles y el retorno futuro de BTC.

El objetivo no es maximizar la precisión predictiva, sino determinar si las variables seleccionadas contienen información suficiente para superar los modelos de referencia.

## 8.1. Entrenamiento del modelo

El modelo se ajusta utilizando exclusivamente las observaciones del conjunto de entrenamiento, respetando la separación temporal establecida previamente.

In [21]:
linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](7,)","[ 0.03, 0.06, 0. ,...,-0. , 0. , 0. ]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](7,)","['btc_return_lag1','btc_return_lag2','btc_volatility_7d_lag1',..., 'dxy_return_lag1','gold_return_lag1','vix_close_lag1']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-0.001593
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,4


## 8.2. Predicción sobre el conjunto de prueba

Una vez entrenado, el modelo genera predicciones sobre observaciones que no fueron utilizadas durante el entrenamiento.

In [22]:
linear_pred = linear_model.predict(
    X_test
)

In [23]:
print(linear_pred[:10])

[-0.00125613  0.0002199   0.00130669  0.00200068  0.00029383  0.00133618
  0.00213645  0.00305359  0.00064323  0.00047285]


## 8.3. Evaluación del modelo lineal

Se evalúa el desempeño de la regresión lineal utilizando las mismas métricas empleadas para los modelos baseline.

Esto permite determinar si la información contenida en las siete variables explicativas aporta capacidad predictiva adicional respecto de las estrategias de referencia.

In [24]:
mae_linear = mean_absolute_error(
    y_test,
    linear_pred
)

rmse_linear = np.sqrt(
    mean_squared_error(
        y_test,
        linear_pred
    )
)

r2_linear = r2_score(
    y_test,
    linear_pred
)

In [25]:
print("=== REGRESIÓN LINEAL ===")
print()

print(f"MAE: {mae_linear:.6f}")
print(f"RMSE: {rmse_linear:.6f}")
print(f"R²: {r2_linear:.6f}")

=== REGRESIÓN LINEAL ===

MAE: 0.018409
RMSE: 0.025659
R²: -0.006929


# 9. Comparación de modelos

Se comparan los resultados obtenidos por los tres enfoques evaluados durante la Fase 1.

El objetivo es determinar si las variables construidas aportan información adicional respecto de estrategias de referencia extremadamente simples.

In [26]:
comparison_df = pd.DataFrame(
    {
        "Modelo": [
            "Baseline Naive",
            "Baseline Media Histórica",
            "Regresión Lineal"
        ],
        "MAE": [
            mae_naive,
            mae_mean,
            mae_linear
        ],
        "RMSE": [
            rmse_naive,
            rmse_mean,
            rmse_linear
        ],
        "R²": [
            r2_naive,
            r2_mean,
            r2_linear
        ]
    }
)

comparison_df

,Modelo,MAE,RMSE,R²
0,Baseline Naive,0.028166,0.037965,-1.204368
1,Baseline Media Histórica,0.018624,0.025780,-0.016449
2,Regresión Lineal,0.018409,0.025659,-0.006929


## Interpretación de resultados

El baseline naive, basado en la persistencia de los retornos, presentó el peor desempeño entre los modelos evaluados. Esto sugiere que la memoria de muy corto plazo, considerada de manera aislada, no aporta capacidad predictiva sobre el retorno diario futuro de BTC.

Por el contrario, el baseline de media histórica constituyó una referencia considerablemente más difícil de superar, obteniendo resultados claramente superiores a los del modelo de persistencia.

La regresión lineal logró mejorar las tres métricas respecto del baseline de media histórica, lo que indica que las variables construidas durante la Fase 1 contienen información relevante para explicar parcialmente el comportamiento futuro de BTC.

Sin embargo, las mejoras obtenidas fueron reducidas, lo que resulta consistente con la naturaleza altamente volátil y ruidosa de los retornos financieros.

# 10. Conclusiones

Durante este notebook se evaluó la capacidad predictiva de las variables construidas en la Fase 1 mediante dos modelos baseline y una regresión lineal.

El baseline de persistencia resultó claramente inferior a los restantes enfoques, evidenciando que la repetición simple del último retorno no constituye una estrategia adecuada para describir el comportamiento diario de BTC.

La regresión lineal consiguió superar, aunque por un margen reducido, al baseline de media histórica en las tres métricas consideradas.

En particular, el modelo obtuvo un coeficiente de determinación R² = -0.0069, superior al R² = -0.0164 del baseline de media histórica.

Si bien ambos valores permanecen próximos a cero, la mejora observada sugiere la existencia de una señal estadística débil en las variables consideradas.

En consecuencia, los resultados obtenidos no permiten afirmar la existencia de relaciones predictivas fuertes, pero sí justifican la continuación del análisis mediante modelos más flexibles y una expansión del conjunto de variables.

Por lo tanto, la evidencia obtenida en esta primera etapa resulta compatible con la hipótesis de que el comportamiento de BTC presenta una elevada componente aleatoria, coexistiendo con señales de baja intensidad susceptibles de ser exploradas mediante técnicas adicionales.

## Próxima etapa

Los resultados obtenidos en la Fase 1 justifican avanzar hacia una segunda etapa de modelado.

En el Notebook 09 se incorporarán nuevas variables, se explorarán modelos más complejos y se evaluará la estabilidad de los resultados obtenidos, con el objetivo de determinar si la información presente en los datos puede ser aprovechada mediante enfoques más sofisticados.

# 11. Exportación de resultados

Se exporta una tabla resumen con las métricas obtenidas por cada uno de los modelos evaluados durante la Fase 1.

Esta tabla servirá como referencia para las comparaciones posteriores realizadas en la Fase 2.

In [31]:
comparison_df.to_csv(
    os.path.join(
        PROJECT_PATH,
        "data",
        "processed",
        "model_comparison_phase1.csv"
    ),
    index=False
)

print("Tabla de comparación exportada correctamente.")

Tabla de comparación exportada correctamente.
